In [ ]:
from transformers import AutoTokenizer, pipeline, AutoModelForCausalLM
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
import numpy as np
from datasets import load_dataset, Dataset
import sys
sys.path.append("/data/chaojian/representation-engineering")
from repe import repe_pipeline_registry, WrappedReadingVecModel
repe_pipeline_registry()

In [ ]:
ds = load_dataset('argilla/ultrafeedback-binarized-preferences-cleaned')


In [ ]:
ds_evol = ds.filter(lambda x: x['source']=='evol_instruct')

In [ ]:
ds_evol

### Helpful v.s. Unhelpful

In [ ]:
ds_evol['train']['chosen'][0][0]

In [ ]:
# orig_template = "{instruction}{response}"
# # control_templates = [
# #     # "Pretend you're a {type} person giving a response.", 
# #     # "Make your response as {type} as possible.",
# #     # "Give a response that is {type}.",
# #     # "Generate a response in a {type} way.",
# # ]
# pos_template = "{instruction} {type} {response}"
# neg_template = "{instruction} {type} {response}"

pos_s = "Pretend you are a helpful assistant. Give me a helpful answer. "
neg_s = "Pretend you are a helpfuless assistant. Give me a helpfuless answer. "

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('/data/chaojian/Llama-2-7b-chat-hf')


# 二维列表，每个元素长度为2，第一个元素为prompt，第二个元素为response
formatted_text_chosen = [tokenizer.apply_chat_template(conv, tokenize=False).split('[/INST]') for conv in ds_evol['train']['chosen']]
formatted_text_rejected = [tokenizer.apply_chat_template(conv, tokenize=False).split('[/INST]') for conv in ds_evol['train']['rejected']]    


In [ ]:
formatted_text_chosen_prompt = [prompt[0] + pos_s for prompt in formatted_text_chosen]
formatted_text_chosen_response = ['[/INST]' + prompt[1] for prompt in formatted_text_chosen]

formatted_text_rejected_prompt = [prompt[0] + neg_s for prompt in formatted_text_rejected]
formatted_text_rejected_response = ['[/INST]' + prompt[1] for prompt in formatted_text_rejected]        

In [ ]:
formatted_text_rejected_prompt[:5]

In [ ]:
tokenizer.pad_token = tokenizer.unk_token

tokenized_chosen_prompt = tokenizer.batch_encode_plus(formatted_text_chosen_prompt, return_tensors="pt", padding=True,)
tokenized_chosen_response = tokenizer.batch_encode_plus(formatted_text_chosen_response, return_tensors="pt", padding=True,)

In [ ]:
np.percentile(tokenized_chosen_prompt['attention_mask'].sum(dim=1).tolist(), 90), np.percentile(tokenized_chosen_response['attention_mask'].sum(dim=1).tolist(), 90)

In [ ]:
np.mean(tokenized_chosen_prompt['attention_mask'].sum(dim=1).tolist()), np.mean(tokenized_chosen_response['attention_mask'].sum(dim=1).tolist())

### Truthful v.s. untruthful

In [ ]:
ds_truth_qa = ds.filter(lambda x: x['source']=='false_qa')
ds_truth_qa

In [ ]:
pos_s = "Pretend you are a truthful assistant. Give me a truthful answer. "
neg_s = "Pretend you are an untruthful  assistant. Give me an untruthful answer. "



In [ ]:
tokenizer = AutoTokenizer.from_pretrained('/data/chaojian/Llama-2-7b-chat-hf')

formatted_text_chosen = [tokenizer.apply_chat_template(conv, tokenize=False).split('[/INST]') for conv in ds_truth_qa['train']['chosen']]
formatted_text_rejected = [tokenizer.apply_chat_template(conv, tokenize=False).split('[/INST]') for conv in ds_truth_qa['train']['rejected']]

In [ ]:
formatted_text_chosen_prompt = [prompt[0] + pos_s for prompt in formatted_text_chosen]
formatted_text_chosen_response = ['[/INST]' + prompt[1] for prompt in formatted_text_chosen]

formatted_text_rejected_prompt = [prompt[0] + neg_s for prompt in formatted_text_rejected]
formatted_text_rejected_response = ['[/INST]' + prompt[1] for prompt in formatted_text_rejected]

In [ ]:
len(formatted_text_chosen_prompt)

In [ ]:
tokenizer.pad_token = tokenizer.unk_token

tokenized_chosen_prompt = tokenizer.batch_encode_plus(formatted_text_chosen_prompt, return_tensors="pt", padding=True,)
tokenized_chosen_response = tokenizer.batch_encode_plus(formatted_text_chosen_response, return_tensors="pt", padding=True,)

In [ ]:
np.percentile(tokenized_chosen_prompt['attention_mask'].sum(dim=1).tolist(), 90), np.percentile(tokenized_chosen_response['attention_mask'].sum(dim=1).tolist(), 90)

In [ ]:
np.mean(tokenized_chosen_prompt['attention_mask'].sum(dim=1).tolist()), np.mean(tokenized_chosen_response['attention_mask'].sum(dim=1).tolist())

### Safety v.s. toxic

In [ ]:
# Advbench
import json
all_data = []

for i in range(6):
    file_name = f"/data/chaojian/Multi-alignment/dataset/toxicity_pairwise/split_{i}.jsonl"
    with open(file_name, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            data = json.loads(line)
            all_data.append(data)

all_data[:5]



In [ ]:
from datasets import Dataset, load_dataset, load_from_disk

ds = load_from_disk('/data/chaojian/Multi-alignment/dataset/toxicity_pairwise')
ds



In [ ]:
ds[0]['prompt_text']

In [ ]:
ds[0]['unpert_gen_text']

In [ ]:
ds[0]['pert_gen_text']

In [ ]:
pos_s = "[INST] Pretend you are a harmless assistant. Give me a halmless statement. "
neg_s = "[INST] Pretend you are a harmful assistant. Give me a harmful statement. "

In [ ]:
formatted_text_chosen = ['[/INST]' + cov for cov in ds['unpert_gen_text']]
formatted_text_rejected = ['[/INST]' + cov for cov in ds['pert_gen_text']]


In [ ]:
formatted_text_chosen

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('/data/chaojian/Llama-2-7b-chat-hf')
tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = 'left'
tokinized_prompt = tokenizer([pos_s, neg_s], padding='longest', return_tensors='pt')

In [ ]:
tokinized_prompt['attention_mask'].size()

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('/data/chaojian/Llama-2-7b-chat-hf')

tokenizer.pad_token = tokenizer.unk_token

tokenized_chosen_prompt = tokenizer.batch_encode_plus(formatted_text_chosen, return_tensors="pt", padding=True,)
tokenized_chosen_response = tokenizer.batch_encode_plus(formatted_text_rejected, return_tensors="pt", padding=True,)


In [ ]:
tokenized_chosen_prompt['attention_mask'].sum(dim=1).max(), tokenized_chosen_response['attention_mask'].sum(dim=1).max()

In [ ]:
import numpy as np
np.percentile(tokenized_chosen_prompt['attention_mask'].sum(dim=1).tolist(), 100), np.percentile(tokenized_chosen_response['attention_mask'].sum(dim=1).tolist(), 100)

In [ ]:
np.mean(tokenized_chosen_prompt['attention_mask'].sum(dim=1).tolist()), np.mean(tokenized_chosen_response['attention_mask'].sum(dim=1).tolist())

In [ ]:
import torch
toxic_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_toxic_hidden_states.pth")
toxic_hidden[21].shape

### Fairness v.s. Bias

In [ ]:
from datasets import load_dataset

ds = load_dataset("McGill-NLP/stereoset", "intrasentence")

In [ ]:
ds

In [ ]:
label = ['anti-stereotype', 'stereotype', 'unrelated']

In [ ]:
bias_type = ds['validation']['bias_type']

In [ ]:
sentences = ds['validation']['sentences'][1]['sentence']
sentences

In [ ]:
labels = ds['validation']['sentences'][1]['gold_label']
labels

In [ ]:
for sentences, labels in zip(sentences, labels):
    print(sentences, labels)

In [ ]:
sentence_with_labels = [zip(statement['sentence'], statement['gold_label'])for statement in ds['validation']['sentences']]

anti_stereotype = []
stereotype = []
unrelated = []
for sentence in sentence_with_labels:
    for sent, label in sentence:
        if label == 0:
            anti_stereotype.append(sent)
        elif label == 1:
            stereotype.append(sent)
        else:
            unrelated.append(sent)



In [ ]:
target = ds['validation']['target']

In [ ]:
ds_bias = Dataset.from_dict({'bias_type': bias_type, 'target':target,'anti_stereotype': anti_stereotype, 'stereotype': stereotype})

In [ ]:
ds_bias

In [ ]:
ds_bias.save_to_disk('/data/chaojian/Multi-alignment/dataset/bias_dataset')

In [ ]:
pos_s = '[INST] Pretend you are a fair assistant. Give me a fair statement about {target}'
neg_s = '[INST] Pretend you are a biased assistant. Give me a biased statement about {target}'


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("/data/chaojian/Llama-2-7b-chat-hf", use_fast=False)
tokenizer.pad_token = tokenizer.unk_token

tokenizer([pos_s, neg_s], padding='longest',return_tensors='pt')['attention_mask'].shape

In [ ]:
chosen_response = ['[/INST] '+ x['anti_stereotype'] for x in ds_bias]
rejected_response = ['[/INST] '+ x['stereotype'] for x in ds_bias]
tokenized_chosen_response = tokenizer.batch_encode_plus(chosen_response, return_tensors="pt", padding=True,)
tokenized_rejected_response = tokenizer.batch_encode_plus(rejected_response, return_tensors="pt", padding=True,)

In [ ]:
chosen_response

In [ ]:
np.percentile(tokenized_chosen_response['attention_mask'].sum(dim=1), 100), np.percentile(tokenized_rejected_response['attention_mask'].sum(dim=1), 100),

In [ ]:
tokenized_chosen_response['attention_mask'].sum(dim=1).max()

### Vacabulary Space

In [ ]:
model_path = '/data/chaojian/Llama-2-7b-chat-hf'

model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map='cuda:0')



In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.unk_token

In [ ]:
a = model.lm_head
b = torch.rand(2,4096).to(model.device)

In [ ]:
a(b)

In [ ]:
vocab_ranking = torch.matmul(a.weight, b[0])
sorted_token_ids = np.argsort(vocab_ranking.detach().cpu().numpy())[::-1]

In [ ]:
sorted_tokens = [tokenizer.decode(x).strip() for x in sorted_token_ids[:10]]
sorted_tokens

In [ ]:
def project_into_vocabluary(vector, E, tokenizer, top_k=50, bottom_k=-1):
    """
    Project a vector into the vocabulary space and return the top_k tokens.
    :param vector: D dimensional vector
    :param E: Language model embedding matrix (V, D)
    :param tokenizer: Model tokenizer
    :param top_k: How many top tokens to return
    :param bottom_k: How many bottom tokens to return. If -1, return top_k tokens
    :return:
    """

    # 源码是torch.float32
    vector = vector.to(torch.float32).to('cuda')
    E = E.to(torch.float32).to('cuda')


    vocab_ranking = torch.matmul(E, vector)     # (V,)
    sorted_token_ids = np.argsort(vocab_ranking.detach().cpu().numpy())[::-1]  # Descending order
    if bottom_k == -1:
        sorted_tokens = [tokenizer.decode(x).strip() for x in sorted_token_ids[:top_k]]
        # logging.debug([(sorted_token_ids[i], sorted_tokens[i], vocab_ranking[sorted_token_ids[i]].item()) for i in range(top_k)])
    else :
        sorted_tokens = [tokenizer.decode(x).strip() for x in sorted_token_ids[-bottom_k:][::-1]]  # Least score to most score
    return sorted_tokens


In [ ]:
toxic_hidden = torch.load('/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_toxic_hidden_states.pth')
help_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_helpful_hidden_states.pth")
truth1_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_truthful_hidden_states.pth")
fair_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_bias_hidden_states.pth")
truth2_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_truthfulqa_hidden_states.pth")

In [ ]:
layer = 32

In [ ]:
toxic_last_layer_mean = toxic_hidden[layer].mean(dim=0)
toxic_last_layer_mean.shape
filter = help_hidden[layer][~help_hidden[layer].isnan().any(dim=1)]
help_last_layer = filter[~filter.isinf().any(dim=1)]
help_last_layer_mean = help_last_layer.mean(dim=0)
filter = truth1_hidden[layer][~truth1_hidden[layer].isnan().any(dim=1)]
truth_last_layer= filter[~filter.isinf().any(dim=1)]
truth_last_layer_mean = truth_last_layer.mean(dim=0)
truth_last_layer_mean
filter = truth2_hidden[layer][~truth2_hidden[layer].isnan().any(dim=1)]
truth2_last_layer= filter[~filter.isinf().any(dim=1)]
truth2_last_layer_mean = truth2_last_layer.mean(dim=0)
truth2_last_layer_mean
filter = fair_hidden[layer][~fair_hidden[layer].isnan().any(dim=1)]
fair_last_layer= filter[~filter.isinf().any(dim=1)]
fair_last_layer_mean = fair_last_layer.mean(dim=0)
fair_last_layer_mean

### Cosine Similarity

#### Mean Vector

In [ ]:
torch.cosine_similarity(truth_last_layer_mean, help_last_layer_mean, dim=0)

In [ ]:
torch.cosine_similarity(toxic_last_layer_mean, help_last_layer_mean, dim=0)

In [ ]:
torch.cosine_similarity(toxic_last_layer_mean, truth_last_layer_mean, dim=0)

In [ ]:
(
torch.cosine_similarity(fair_last_layer_mean, help_last_layer_mean, dim=0), \
torch.cosine_similarity(fair_last_layer_mean, truth_last_layer_mean, dim=0), \
torch.cosine_similarity(fair_last_layer_mean, toxic_last_layer_mean, dim=0), \
)

In [ ]:
(
    torch.cosine_similarity(truth2_last_layer_mean, truth_last_layer_mean, dim=0),
    torch.cosine_similarity(truth2_last_layer_mean, toxic_last_layer_mean, dim=0),
    torch.cosine_similarity(truth2_last_layer_mean, help_last_layer_mean, dim=0),
    torch.cosine_similarity(truth2_last_layer_mean, fair_last_layer_mean, dim=0),
)

#### Principal Component

In [ ]:
toxic_pca = torch.pca_lowrank(toxic_hidden[layer].to(torch.float32), q=2, niter=50)[2][:,0]
helpful_pca = torch.pca_lowrank(help_last_layer.to(torch.float32), q=2, niter=50)[2][:,0]
truthful1_pca = torch.pca_lowrank(truth_last_layer.to(torch.float32), q=2, niter=50)[2][:,0]
fair_pca = torch.pca_lowrank(fair_last_layer.to(torch.float32), q=2, niter=50)[2][:,0]
truthful2_pca = torch.pca_lowrank(truth2_last_layer.to(torch.float32), q=2, niter=50)[2][:,0]

In [ ]:
torch.cosine_similarity(toxic_pca, helpful_pca, dim=0), \
torch.cosine_similarity(toxic_pca, truthful1_pca, dim=0), \
torch.cosine_similarity(truthful1_pca, helpful_pca, dim=0),\
torch.cosine_similarity(fair_pca, toxic_pca, dim=0), \
torch.cosine_similarity(fair_pca, helpful_pca, dim=0), \
torch.cosine_similarity(fair_pca, truthful1_pca, dim=0),\
torch.cosine_similarity(truthful2_pca, truthful1_pca, dim=0),\
torch.cosine_similarity(truthful2_pca, helpful_pca, dim=0),\
torch.cosine_similarity(truthful2_pca, fair_pca, dim=0),\
torch.cosine_similarity(truthful2_pca, toxic_pca, dim=0)


In [ ]:
for key in model.state_dict():
    if 'weight' in key and 'mlp' in key:
        layer_num = int(key.split('.')[2])
        print(key)
        print(layer_num)

In [ ]:
a = torch.rand(2,8,4)
a[:, -3:]

In [ ]:
model = AutoModelForCausalLM.from_pretrained("/data/chaojian/Llama-2-7b-chat-hf", torch_dtype=torch.float16, device_map="cuda:0")
tokenizer = AutoTokenizer.from_pretrained("/data/chaojian/Llama-2-7b-chat-hf")


In [ ]:
tokenizer.pad_token = tokenizer.unk_token
text = ['I love nlp', 'I love cv', 'I love machine learning','fuck off']


inputs = tokenizer(text, return_tensors="pt", padding=True)

In [ ]:
b = inputs['attention_mask'][:, -3:].repeat(2, 1, 1).unsqueeze(-1)

In [ ]:
# 每一个[1]会在相乘的时候广播成为hidden_dim
b[0][0]

In [ ]:
import numpy as np



## Different DataSet

In [ ]:
ds_truthful_qa = load_dataset("truthful_qa", "generation")

In [ ]:
ds_truthful_qa['validation']['question'][1]

In [ ]:
ds_truthful_qa['validation']['correct_answers'][2]

In [ ]:
ds_truthful_qa['validation']['incorrect_answers'][2]

In [ ]:
truthful_qa = {}
questions = []
corret_answers = []
incorrect_answers = []
for i in range(len(ds_truthful_qa['validation']['question'])):
    for corr, incorr in zip(ds_truthful_qa['validation']['correct_answers'][i], ds_truthful_qa['validation']['incorrect_answers'][i]):
        questions.append(ds_truthful_qa['validation']['question'][i])
        corret_answers.append(corr)
        incorrect_answers.append(incorr)




In [ ]:
truthful_qa = {'questions':questions, 'correct_answers':corret_answers, 'incorrect_answers':incorrect_answers}

In [ ]:
ds_truth_qa = Dataset.from_dict(truthful_qa)
ds_truth_qa

In [ ]:
ds_truth_qa.save_to_disk('/data/chaojian/Multi-alignment/dataset/truthful_qa')

In [ ]:
pos_s = "Pretend you are a truthful assistant. Give me a truthful answer. "
neg_s = "Pretend you are an untruthful assistant. Give me an untruthful answer. " 



formatted_text_chosen_prompt = ['[INST]' + pos_s +  cov for cov in ds_truth_qa['questions']]
formatted_text_rejected_prompt = ['[INST]' + neg_s + cov for cov in ds_truth_qa['questions']]

formatted_text_chosen = ['[/INST]' + cov for cov in ds_truth_qa['correct_answers']]
formatted_text_rejected = ['[/INST]' + cov for cov in ds_truth_qa['incorrect_answers']]



In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('/data/chaojian/Llama-2-7b-chat-hf')

tokenizer.pad_token = tokenizer.unk_token

tokenized_chosen_prompt = tokenizer.batch_encode_plus(formatted_text_chosen_prompt, return_tensors="pt", padding=True,)
tokenized_rejected_prompt = tokenizer.batch_encode_plus(formatted_text_rejected_prompt, return_tensors="pt", padding=True,)

tokenized_chosen_response = tokenizer.batch_encode_plus(formatted_text_chosen, return_tensors="pt", padding=True,)
tokenized_rejected_response = tokenizer.batch_encode_plus(formatted_text_rejected, return_tensors="pt", padding=True,)

In [ ]:
(tokenized_chosen_prompt['attention_mask'].sum(1).max(), tokenized_rejected_prompt['attention_mask'].sum(1).max())

In [ ]:
(tokenized_chosen_response['attention_mask'].sum(1).max(), tokenized_rejected_response['attention_mask'].sum(1).max())

## Different Layers

In [ ]:
import torch

toxic_hidden = torch.load('/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_toxic_hidden_states.pth')
help_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_helpful_hidden_states.pth")
truth1_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_falseqa_hidden_states.pth")
fair_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_bias_hidden_states.pth")
truth2_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_truthfulqa_hidden_states.pth")

In [ ]:
layer = 32
toxic_last_layer_mean = toxic_hidden[layer].mean(dim=0)

filter = help_hidden[layer][~help_hidden[layer].isnan().any(dim=1)]
help_last_layer = filter[~filter.isinf().any(dim=1)]
help_last_layer_mean = help_last_layer.mean(dim=0)

filter = truth1_hidden[layer][~truth1_hidden[layer].isnan().any(dim=1)]
truth_last_layer= filter[~filter.isinf().any(dim=1)]
truth_last_layer_mean = truth_last_layer.mean(dim=0)

filter = truth2_hidden[layer][~truth2_hidden[layer].isnan().any(dim=1)]
truth2_last_layer= filter[~filter.isinf().any(dim=1)]
truth2_last_layer_mean = truth2_last_layer.mean(dim=0)

filter = fair_hidden[layer][~fair_hidden[layer].isnan().any(dim=1)]
fair_last_layer= filter[~filter.isinf().any(dim=1)]
fair_last_layer_mean = fair_last_layer.mean(dim=0)


In [ ]:
q = 4
pca = 0
toxic_pca = torch.pca_lowrank(toxic_hidden[layer].to(torch.float32), q=q, niter=100)[2][:,pca]
helpful_pca = torch.pca_lowrank(help_last_layer.to(torch.float32), q=q, niter=100)[2][:,pca]
truthful1_pca = torch.pca_lowrank(truth_last_layer.to(torch.float32), q=q, niter=100)[2][:,pca]
fair_pca = torch.pca_lowrank(fair_last_layer.to(torch.float32), q=q, niter=100)[2][:,pca]
truthful2_pca = torch.pca_lowrank(truth2_last_layer.to(torch.float32), q=q, niter=100)[2][:,pca]

In [ ]:
pca_matrix = torch.stack([toxic_pca, helpful_pca, truthful1_pca, truthful2_pca, fair_pca])
pca_matrix_1 = pca_matrix.unsqueeze(1)
pca_matrix_2 = pca_matrix.unsqueeze(0)
cos_matrix = torch.cosine_similarity(pca_matrix_1, pca_matrix_2, dim=-1)
import matplotlib.pyplot as plt
import seaborn as sns
labels = ['toxic', 'helpful', 'truth1', 'truth2','fair']

plt.figure(figsize=(8, 6))
sns.heatmap(
    cos_matrix,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    square=True,
    xticklabels=labels,
    yticklabels=labels,
    linewidths=0.5
)
plt.title(f"Cosine Similarity Matrix: Layer {layer}, pca {pca}")

## Coarse Grained Rep

In [ ]:
from datasets import load_dataset, load_from_disk, Dataset


ds1 = load_dataset('argilla/ultrafeedback-binarized-preferences-cleaned')

ds_evol = ds1.filter(lambda x: x['source']=='evol_instruct')

ds_evol

In [ ]:
ds2 = load_from_disk('/data/chaojian/Multi-alignment/dataset/bias_dataset')
ds2


In [ ]:
ds3 = load_from_disk('/data/chaojian/Multi-alignment/dataset/toxicity_pairwise')
ds3

In [ ]:
ds4 = load_from_disk('/data/chaojian/Multi-alignment/dataset/truthful_qa')
ds4

In [ ]:
ds5 = ds1.filter(lambda x: x['source']=='false_qa')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("/data/chaojian/Llama-2-7b-chat-hf")

In [ ]:
helpful_chosen = ds_evol['train'].to_dict()['chosen']
helpful_rejected = ds_evol['train'].to_dict()['rejected']

falseqa_chosen = ds5['train'].to_dict()['chosen']
falseqa_rejected = ds5['train'].to_dict()['rejected']

In [ ]:
[[1,2,3],[3,4,5]] + [[4,2,1],[3,4,5]]

In [ ]:
ds2['anti_stereotype']

In [ ]:
template = "make a statement about {topic}"

prompt = [template.format(topic=topic) for topic in ds2['target']]

chosen_response = ds2['anti_stereotype']
rejected_response = ds2['stereotype']



In [ ]:
fair_text_chosen = [[{'content': prompt, 'role': 'user'},
                   {'content': completion, 'role': 'assistant'}] for prompt, completion in zip(prompt, chosen_response)]

bias_text_rejected = [[{'content': prompt, 'role': 'user'},
                   {'content': completion, 'role': 'assistant'}] for prompt, completion in zip(prompt, rejected_response)]



In [ ]:
template1 = "make a harmless statement."
template2 = "make a harmful statement."

untoxic_text = [[{'content': template1, 'role': 'user'},
                   {'content': completion, 'role': 'assistant'}] for completion in ds3['unpert_gen_text']]

toxic_text = [[{'content': template2, 'role': 'user'},
                   {'content': completion, 'role': 'assistant'}] for completion in ds3['pert_gen_text']]


In [ ]:
ds4

In [ ]:
truthful_answer = [[{'content': prompt, 'role': 'user'},
                   {'content': completion, 'role': 'assistant'}] for prompt, completion in zip(ds4['questions'], ds4['correct_answers'])]
untruthful_answer = [[{'content': prompt, 'role': 'user'},
                   {'content': completion, 'role': 'assistant'}] for prompt, completion in zip(ds4['questions'], ds4['incorrect_answers'])]

In [ ]:
preference_text = helpful_chosen + untoxic_text + truthful_answer + falseqa_chosen + fair_text_chosen
rejected_text = helpful_rejected + toxic_text + untruthful_answer + falseqa_rejected + bias_text_rejected

In [ ]:
ds_all = Dataset.from_dict({'preference':preference_text,'rejected': rejected_text})
ds_all.save_to_disk('/data/chaojian/Multi-alignment/dataset/coarse_grained')

In [ ]:
ds_all

In [ ]:
formatted_text_chosen = [tokenizer.apply_chat_template(conv, tokenize=False).split('[/INST]') for conv in ds_all['preference']]
formatted_text_rejected = [tokenizer.apply_chat_template(conv, tokenize=False).split('[/INST]') for conv in ds_all['rejected']]

In [ ]:
formatted_text_chosen_prompt = [prompt[0] for prompt in formatted_text_chosen]
formatted_text_chosen_response = ['[/INST]'+prompt[1] for prompt in formatted_text_chosen]

formatted_text_rejected_prompt = [prompt[0] for prompt in formatted_text_rejected]
formatted_text_rejected_response = ['[/INST]'+prompt[1] for prompt in formatted_text_rejected]


In [ ]:
tokenizer.pad_token = tokenizer.unk_token

tokenized_chosen_prompt = tokenizer.batch_encode_plus(formatted_text_chosen_prompt, return_tensors="pt", padding=True,)
tokenized_rejected_prompt = tokenizer.batch_encode_plus(formatted_text_rejected_prompt, return_tensors="pt", padding=True,)

tokenized_chosen_response = tokenizer.batch_encode_plus(formatted_text_chosen_response, return_tensors="pt", padding=True,)
tokenized_rejected_response = tokenizer.batch_encode_plus(formatted_text_rejected_response, return_tensors="pt", padding=True,)

In [ ]:
import numpy as np
(tokenized_chosen_prompt['attention_mask'].sum(1).max(), tokenized_rejected_prompt['attention_mask'].sum(1).max())

In [ ]:
np.percentile(tokenized_chosen_prompt['attention_mask'].sum(1), 90), np.percentile(tokenized_rejected_prompt['attention_mask'].sum(1), 90)

In [ ]:
tokenized_chosen_response['attention_mask'].sum(1).max(), tokenized_rejected_response['attention_mask'].sum(1).max()

In [ ]:
np.percentile(tokenized_chosen_response['attention_mask'].sum(1), 90), np.percentile(tokenized_rejected_response['attention_mask'].sum(1), 85)

In [ ]:
import torch

toxic_hidden = torch.load('/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_toxic_hidden_states.pth')
help_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_helpful_hidden_states.pth")
truth1_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_falseqa_hidden_states.pth")
fair_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_bias_hidden_states.pth")
truth2_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_truthfulqa_hidden_states.pth")
coarse_grained_hidden = torch.load("/data/chaojian/Multi-alignment/llama2_preference_hidden_states/final_coarse_grained2_hidden_states.pth")

In [ ]:
layer = 32
toxic_last_layer_mean = toxic_hidden[layer].mean(dim=0)

filter = help_hidden[layer][~help_hidden[layer].isnan().any(dim=1)]
help_last_layer = filter[~filter.isinf().any(dim=1)]
help_last_layer_mean = help_last_layer.mean(dim=0)

filter = truth1_hidden[layer][~truth1_hidden[layer].isnan().any(dim=1)]
truth_last_layer= filter[~filter.isinf().any(dim=1)]
truth_last_layer_mean = truth_last_layer.mean(dim=0)

filter = truth2_hidden[layer][~truth2_hidden[layer].isnan().any(dim=1)]
truth2_last_layer= filter[~filter.isinf().any(dim=1)]
truth2_last_layer_mean = truth2_last_layer.mean(dim=0)

filter = fair_hidden[layer][~fair_hidden[layer].isnan().any(dim=1)]
fair_last_layer= filter[~filter.isinf().any(dim=1)]
fair_last_layer_mean = fair_last_layer.mean(dim=0)

filter = coarse_grained_hidden[layer][~coarse_grained_hidden[layer].isnan().any(dim=1)]
coarse_last_layer= filter[~filter.isinf().any(dim=1)]
coarse_last_layer_mean = coarse_last_layer.mean(dim=0)

In [ ]:
import numpy as np
toxic_hidden[layer] = toxic_hidden[layer] / np.linalg.norm(toxic_hidden[layer], axis=1, keepdims=True)
help_last_layer = help_last_layer / np.linalg.norm(help_last_layer, axis=1, keepdims=True)
truth_last_layer = truth_last_layer / np.linalg.norm(truth_last_layer, axis=1, keepdims=True)
fair_last_layer = fair_last_layer / np.linalg.norm(fair_last_layer, axis=1, keepdims=True)
truth2_last_layer = truth2_last_layer / np.linalg.norm(truth2_last_layer, axis=1, keepdims=True)
coarse_last_layer = coarse_last_layer / np.linalg.norm(coarse_last_layer, axis=1, keepdims=True)

In [ ]:
toxic_layer_mean = toxic_hidden[layer].mean(dim=0)
help_last_layer_mean = help_last_layer.mean(dim=0)
truth_last_layer_mean = truth_last_layer.mean(dim=0)
truth2_last_layer_mean = truth2_last_layer.mean(dim=0)
fair_last_layer_mean = fair_last_layer.mean(dim=0)
coarse_last_layer_mean = coarse_last_layer.mean(dim=0)


In [ ]:

q = 4
toxic_pca = torch.pca_lowrank(toxic_hidden[layer].to(torch.float32), q=q, niter=50)[2]
helpful_pca = torch.pca_lowrank(help_last_layer.to(torch.float32), q=q, niter=50)[2]
truthful1_pca = torch.pca_lowrank(truth_last_layer.to(torch.float32), q=q, niter=50)[2]
fair_pca = torch.pca_lowrank(fair_last_layer.to(torch.float32), q=q, niter=50)[2]
truthful2_pca = torch.pca_lowrank(truth2_last_layer.to(torch.float32), q=q, niter=50)[2]
coarse_hidden_pca = torch.pca_lowrank(coarse_last_layer.to(torch.float32), q=q, niter=50)[2]

In [ ]:
pca = 3
toxic_pca_n = toxic_pca[:, pca]
helpful_pca_n = helpful_pca[:, pca]
truthful1_pca_n = truthful1_pca[:, pca]
fair_pca_n = fair_pca[:, pca]
truthful2_pca_n = truthful2_pca[:, pca]
coarse_hidden_pca_n = coarse_hidden_pca[:, pca]

In [ ]:
pca_matrix = torch.stack([toxic_pca_n, helpful_pca_n, truthful1_pca_n, truthful2_pca_n, fair_pca_n, coarse_hidden_pca_n])
pca_matrix_1 = pca_matrix.unsqueeze(1)
pca_matrix_2 = pca_matrix.unsqueeze(0)
cos_matrix = torch.cosine_similarity(pca_matrix_1, pca_matrix_2, dim=-1)
import matplotlib.pyplot as plt
import seaborn as sns
labels = ['toxic', 'helpful', 'truth1', 'truth2','fair', 'coarse']

plt.figure(figsize=(8, 6))
sns.heatmap(
    cos_matrix,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    square=True,
    xticklabels=labels,
    yticklabels=labels,
    linewidths=0.5
)
plt.title(f"Cosine Similarity Matrix: Layer {layer}, pca {pca}")

In [ ]:
mean_matrix = torch.stack([toxic_last_layer_mean, help_last_layer_mean, truth_last_layer_mean, truth2_last_layer_mean, fair_last_layer_mean, coarse_last_layer_mean])
mean_matrix_1 = mean_matrix.unsqueeze(1)
mean_matrix_2 = mean_matrix.unsqueeze(0)

cos_matrix = torch.cosine_similarity(mean_matrix_1, mean_matrix_2, dim=-1)
import matplotlib.pyplot as plt
import seaborn as sns
labels = ['toxic', 'helpful', 'truth1', 'truth2','fair', 'coarse']

plt.figure(figsize=(8, 6))
sns.heatmap(
    cos_matrix,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    square=True,
    xticklabels=labels,
    yticklabels=labels,
    linewidths=0.5
)
plt.title(f"Cosine Similarity Matrix: Layer {layer}, Mean")


In [ ]:
torch.cosine_similarity(coarse_hidden_pca_n, helpful_pca_n, dim=0)

In [ ]:
# 回答问题与陈述是否不太一样？
# 没有使用repe的代码，是按照repe的思路自己写的

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = '/data/chaojian/Llama-2-7b-chat-hf'

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="cuda:0", torch_dtype=torch.float16, output_hidden_states=True)  

model.eval()

In [ ]:
texts = ['I love you so much',
         "I don't like you at all",
         "hate you"]

tokenizer.pad_token = tokenizer.unk_token

input_ids = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
input_ids

In [ ]:
output_ids = model(**input_ids)

In [ ]:
last_layer = output_ids['hidden_states'][-1]

In [ ]:
last_layer[0][6:]

In [ ]:
last_layer[1][6:]

In [ ]:
last_layer[2][6:]

In [ ]:
attn_mask = input_ids['attention_mask']

In [ ]:
last_layer.

In [ ]:
new_hidden = attn_mask[:,:,None] * last_layer

In [ ]:
new_hidden[2][3:]

In [ ]:

sum_emb = (last_layer * attn_mask.unsqueeze(-1)).sum(dim=1)
valid = attn_mask.sum(dim=1).clamp(min=1e-9)
   
sum_emb = sum_emb / valid.unsqueeze(-1)

In [ ]:
sum_emb

In [ ]:
attn_mask.unsqueeze(-1).shape

In [ ]:
attn_mask.unsqueeze(-1)